# Unidad 2: Programación Orientada a Objetos (POO)
**Materia:** Programación — Licenciatura en Negocios Digitales (3º año)
**Institución:** Universidad del Museo Social Argentino (UMSA)

---

## Introducción y Contexto de Negocio

A diferencia de los scripts de análisis de datos que son de naturaleza lineal, las plataformas SaaS, las fintech o los portales de e-commerce son sistemas complejos compuestos de entidades que interactúan de forma continua.

La **Programación Orientada a Objetos (POO)** es el paradigma fundamental de desarrollo que nos permite modelar el negocio real en código. Unificando **datos** (atributos) y **comportamiento** (métodos), podemos crear piezas de código modulares, testeables y altamente reutilizables. En esta unidad, aprenderemos a modelar usuarios, carritos, productos y transacciones reales del backend de un negocio digital.

### Objetivos de Aprendizaje:
1. Modelar clases de negocio usando atributos de instancia, de clase y constructores.
2. Implementar los pilares de Abstracción y Encapsulamiento usando atributos privados y el decorador `@property`.
3. Aplicar los conceptos de Herencia y Polimorfismo en el diseño de componentes flexibles (suscripciones, pasarelas de pago).
4. Diseñar interacciones complejas entre múltiples objetos (Usuario -> Carrito -> Productos).


## 1. Clases, Objetos y Atributos

Una **Clase** es la plantilla o molde que describe los atributos y métodos de una entidad. Un **Objeto** es una instancia física creada a partir de esa plantilla.


In [ ]:
class Usuario:
    # Atributo de Clase (compartido por todas las instancias)
    plataforma = "UMSA Digital Hub"

    def __init__(self, nombre: str, email: str):
        # Atributos de Instancia (propios de cada objeto)
        self.nombre = nombre
        self.email = email
        self.activo = True

    # Métodos Especiales (Dunder Methods)
    def __str__(self):
        # Representación amigable para humanos
        return f"{self.nombre} ({self.email})"

    def __repr__(self):
        # Representación técnica para desarrolladores
        return f"Usuario(nombre='{self.nombre}', email='{self.email}')"

    # Método regular
    def suspender(self):
        self.activo = False
        print(f"El usuario {self.nombre} ha sido suspendido temporalmente.")

# Creación de instancias
usuario_1 = Usuario("Paula Ferreyra", "paula@umsa.edu.ar")
usuario_2 = Usuario("Martín Gómez", "martin@umsa.edu.ar")

print("Representación str:", str(usuario_1))
print("Representación repr:", repr(usuario_2))

usuario_2.suspender()
print(f"¿Usuario 2 activo?: {usuario_2.activo}")


## 2. Abstracción y Encapsulamiento con `@property`

El **Encapsulamiento** consiste en ocultar los detalles internos de un objeto y proteger su estado, impidiendo modificaciones directas inválidas.
En Python, marcamos los atributos privados usando doble guión bajo (`__`) y controlamos su lectura y modificación por medio de **getters** y **setters** utilizando el decorador `@property`.


In [ ]:
class Producto:
    def __init__(self, nombre: str, precio_base: float):
        self.nombre = nombre
        # Usamos el setter internamente para aplicar la validación inicial
        self.precio = precio_base

    # Getter: expone el valor de forma controlada
    @property
    def precio(self) -> float:
        return self.__precio

    # Setter: valida las reglas de negocio antes de modificar el dato
    @precio.setter
    def precio(self, nuevo_precio: float):
        if nuevo_precio <= 0:
            raise ValueError("El precio de un producto debe ser estrictamente mayor a 0.")
        self.__precio = nuevo_precio

# Pruebas de encapsulamiento
try:
    prod = Producto("Licencia SaaS Pro", 99.99)
    print(f"Producto creado: {prod.nombre} - Precio: ${prod.precio}")

    # Intentamos cambiar a un precio no permitido
    prod.precio = -10.0
except ValueError as e:
    print(f"Error detectado y controlado: {e}")


## 3. Herencia y Polimorfismo

- **Herencia**: Permite que una clase hija adquiera los atributos y métodos de una clase padre, facilitando la reutilización del código.
- **Polimorfismo**: Permite que diferentes clases expongan el mismo nombre de método pero lo implementen de formas distintas (comportamientos especializados).


In [ ]:
# Clase Padre
class Suscripcion:
    def __init__(self, precio_base: float):
        self.precio_base = precio_base

    def calcular_tarifa(self) -> float:
        # Tarifa estándar
        return self.precio_base

# Clase Hija 1: Hereda de Suscripcion
class SuscripcionAnual(Suscripcion):
    def calcular_tarifa(self) -> float:
        # Polimorfismo: aplica un 20% de descuento por pago adelantado
        return (self.precio_base * 12) * 0.8

# Clase Hija 2: Hereda de Suscripcion
class SuscripcionFamiliar(Suscripcion):
    def __init__(self, precio_base: float, miembros: int):
        # Llamamos al constructor del padre
        super().__init__(precio_base)
        self.miembros = miembros

    def calcular_tarifa(self) -> float:
        # Costo base multiplicado por miembros, con 10% de descuento
        costo_total = self.precio_base * self.miembros
        return costo_total * 0.9

# Ejecutando comportamiento polimórfico
planes = [
    Suscripcion(50.0),                     # Mensual simple
    SuscripcionAnual(50.0),                # Anual con descuento
    SuscripcionFamiliar(50.0, 4)           # Familiar con 4 personas
]

for idx, plan in enumerate(planes, 1):
    print(f"Plan {idx} - Costo final: ${plan.calcular_tarifa():.2f}")


## 4. Modelado Completo de un Flujo de E-commerce

A continuación veremos un ejemplo integrado donde múltiples clases colaboran para modelar un flujo de checkout completo de comercio electrónico.


In [ ]:
class ItemCarrito:
    def __init__(self, producto: Producto, cantidad: int):
        self.producto = producto
        self.cantidad = cantidad

    @property
    def subtotal(self) -> float:
        return self.producto.precio * self.cantidad

class Carrito:
    def __init__(self):
        self.items = []

    def agregar_item(self, producto: Producto, cantidad: int = 1):
        # Buscamos si ya está el producto
        for item in self.items:
            if item.producto.nombre == producto.nombre:
                item.cantidad += cantidad
                return
        self.items.append(ItemCarrito(producto, cantidad))

    @property
    def total(self) -> float:
        return sum(item.subtotal for item in self.items)

    def mostrar_detalle(self):
        print("=== Detalle del Carrito ===")
        for item in self.items:
            print(f" - {item.producto.nombre} x{item.cantidad}: ${item.subtotal:.2f}")
        print(f"Total: ${self.total:.2f}")

# Simulación de compra
prod1 = Producto("Suscripción CRM", 45.0)
prod2 = Producto("Módulo Analítica", 25.0)

carrito = Carrito()
carrito.agregar_item(prod1, 1)
carrito.agregar_item(prod2, 2)
carrito.agregar_item(prod1, 1) # Aumenta cantidad de CRM a 2

carrito.mostrar_detalle()


# Unidad 2: Manejo de Excepciones, Módulos y Entornos Virtuales
**Materia:** Programación — Licenciatura en Negocios Digitales (3º año)
**Institución:** Universidad del Museo Social Argentino (UMSA)

---

## 1. Gestión Estructurada de Errores

En entornos de producción, el software debe ser resiliente ante fallos (caídas de red, formatos inválidos, archivos inexistentes).

### Bloque `try-except-else-finally`
- **`try`**: Encapsula el código propenso a fallas.
- **`except`**: Captura y gestiona la excepción sin detener la aplicación.
- **`else`**: Se ejecuta únicamente si **no ocurrió ninguna excepción** en el bloque `try`.
- **`finally`**: Se ejecuta **siempre**, con o sin error (ideal para cerrar recursos o conexiones).

In [1]:
# Definición de Excepción Personalizada de Dominio
class TransaccionInvalidaError(Exception):
    """Excepción lanzada cuando una transacción no cumple las reglas de negocio."""
    def __init__(self, mensaje: str, monto: float):
        super().__init__(mensaje)
        self.monto = monto

def procesar_pago(monto: float, cuenta_activa: bool) -> float:
    """Procesa un pago aplicando reglas de validación de negocio."""
    if not cuenta_activa:
        raise TransaccionInvalidaError("La cuenta del cliente se encuentra inactiva.", monto)
    if monto <= 0:
        raise ValueError("El monto de la transacción debe ser mayor a cero.")
    return monto

# Ejecución controlada con try-except-else-finally
def ejecutar_checkout(monto: float, cuenta_activa: bool) -> None:
    print(f"\n--- Iniciando intento de cobro por ${monto:.2f} ---")
    try:
        resultado = procesar_pago(monto, cuenta_activa)
    except TransaccionInvalidaError as err:
        print(f"[ALERTA DE NEGOCIO] Excepción de dominio: {err} (Monto: ${err.monto:.2f})")
    except ValueError as err:
        print(f"[ERROR DE VALIDACIÓN] Entrada inválida: {err}")
    except Exception as err:
        print(f"[ERROR INESPERADO] Se produjo una falla no contemplada: {err}")
    else:
        print(f"[ÉXITO] Transacción procesada correctamente por ${resultado:.2f}")
    finally:
        print("[AUDITORÍA] Registro de intento finalizado y conexión liberada.")

if __name__ == "__main__":
    ejecutar_checkout(150.0, cuenta_activa=True)   # Flujo Exitoso
    ejecutar_checkout(-50.0, cuenta_activa=True)   # ValueError
    ejecutar_checkout(200.0, cuenta_activa=False)  # TransaccionInvalidaError


--- Iniciando intento de cobro por $150.00 ---
[ÉXITO] Transacción procesada correctamente por $150.00
[AUDITORÍA] Registro de intento finalizado y conexión liberada.

--- Iniciando intento de cobro por $-50.00 ---
[ERROR DE VALIDACIÓN] Entrada inválida: El monto de la transacción debe ser mayor a cero.
[AUDITORÍA] Registro de intento finalizado y conexión liberada.

--- Iniciando intento de cobro por $200.00 ---
[ALERTA DE NEGOCIO] Excepción de dominio: La cuenta del cliente se encuentra inactiva. (Monto: $200.00)
[AUDITORÍA] Registro de intento finalizado y conexión liberada.


## 2. Lectura y Escritura de Archivos Estructurados

Las aplicaciones modernas intercambian información mediante múltiples formatos de datos:
- **CSV**: Intercambio plano e interoperable.
- **JSON**: Payload estándar para APIs Web.
- **YAML**: Archivos de configuración estructurados.
- **Parquet**: Almacenamiento columnar optimizado para analítica de datos a gran escala.

In [2]:
import json
import csv
import pandas as pd

# 1. Escritura y Lectura de JSON
payload_transaccion = {
    "transaccion_id": "TX-9982",
    "cliente": "Sofía Rossi",
    "monto": 1250.50,
    "estado": "completado"
}

with open("transaccion.json", "w", encoding="utf-8") as f:
    json.dump(payload_transaccion, f, indent=4, ensure_ascii=False)

with open("transaccion.json", "r", encoding="utf-8") as f:
    datos_json = json.load(f)
print("Lectura JSON:", datos_json)

# 2. Escritura y Lectura de CSV
registros = [
    ["id", "monto", "metodo"],
    ["TX-101", 150.0, "Tarjeta"],
    ["TX-102", 300.0, "MercadoPago"]
]

with open("ventas.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(registros)

# 3. Conversión y Exportación a Parquet mediante Pandas
df_ventas = pd.read_csv("ventas.csv")
df_ventas.to_parquet("ventas.parquet", index=False)

# Lectura de Parquet
df_parquet = pd.read_parquet("ventas.parquet")
print("\nDataFrame leído desde Parquet:")
print(df_parquet)

Lectura JSON: {'transaccion_id': 'TX-9982', 'cliente': 'Sofía Rossi', 'monto': 1250.5, 'estado': 'completado'}

DataFrame leído desde Parquet:
       id  monto       metodo
0  TX-101  150.0      Tarjeta
1  TX-102  300.0  MercadoPago


## 3. Estructuración de Proyectos y Entornos Virtuales

En desarrollos profesionales, aislar las dependencias mediante entornos virtuales previene conflictos entre paquetes globalmente instalados.

### Gestión de Entornos Virtuales
1. **`venv` (Módulo Nativo de Python):**
   ```bash
   python -m venv .venv
   source .venv/bin/activate  # En Linux/macOS
   .venv\Scripts\activate     # En Windows

mi_proyecto/
│
├── src/
│   └── procesador/
│       ├── __init__.py
│       ├── validador.py
│       └── logger.py
├── tests/
├── .gitignore
├── pyproject.toml
└── README.md

---

#### Bloque 4: Aplicación Práctica (Procesador de Transacciones con Logs)

```markdown
## 4. Aplicación Práctica: Procesador de Transacciones con Logs de Errores

A continuación, implementamos un pipeline completo que valida transacciones en formato JSON, aplica estándares PEP 8, tipeado estricto y registra un archivo de logs (`transacciones.log`).

In [3]:
import logging
from typing import Dict, Any

# Configuración del sistema de Logging nativo
logging.basicConfig(
    filename="transacciones.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    encoding="utf-8"
)

def validar_y_procesar_transaccion(payload: Dict[str, Any]) -> bool:
    """
    Valida la estructura de un payload de transacción y registra logs del proceso.

    :param payload: Diccionario con la información del evento de pago.
    :return: True si la transacción fue procesada con éxito, False en caso contrario.
    """
    tx_id = payload.get("id", "DESCONOCIDO")
    try:
        monto = float(payload["monto"])
        if monto <= 0:
            raise ValueError(f"El monto debe ser estrictamente positivo. Recibido: {monto}")

        logging.info(f"Transacción {tx_id} aprobada exitosamente por ${monto:.2f}.")
        return True

    except KeyError as e:
        logging.error(f"Transacción {tx_id} fallida: Campo requerido ausente {e}")
        return False
    except ValueError as e:
        logging.warning(f"Transacción {tx_id} rechazada por regla de negocio: {e}")
        return False

# Flujo de prueba
if __name__ == "__main__":
    batch_transacciones = [
        {"id": "TX-01", "monto": 450.0},
        {"id": "TX-02", "monto": -100.0},          # Fallo de negocio
        {"id": "TX-03", "detalles": "Sin monto"}   # Fallo de estructura (KeyError)
    ]

    for tx in batch_transacciones:
        validar_y_procesar_transaccion(tx)

    print("Procesamiento completado. Revisa 'transacciones.log' para inspeccionar el historial.")

ERROR:root:Transacción TX-03 fallida: Campo requerido ausente 'monto'


Procesamiento completado. Revisa 'transacciones.log' para inspeccionar el historial.


---

## Desafío Práctico (Trabajo Práctico 2)

**Consigna de Negocio (Modelado de Transacciones):**
Un e-commerce de Negocios Digitales necesita implementar una capa de transacciones con soporte para distintos medios de pago.

1. Diseña una clase `Transaccion` básica que contenga:
   - Atributo privado `__monto` con su respectiva `@property` y validación de que sea mayor a 0.
   - Método `procesar_pago()` que retorne un string `"Procesando transacción general."`
2. Diseña dos clases hijas que hereden de `Transaccion`:
   - `TransaccionMercadoPago`: Debe inicializarse con el `email_pagador` y sobrescribir `procesar_pago()` para que devuelva:
     `"Cobrando $[monto] a [email_pagador] mediante Mercado Pago (QR/Dinero en cuenta)."`
   - `TransaccionStripe`: Debe inicializarse con la `moneda` (ej. `"USD"`, `"EUR"`) y sobrescribir `procesar_pago()` para que devuelva:
     `"Procesando cobro internacional de [moneda] $[monto] con tarjeta de crédito vía Stripe."`
3. Implementa una clase `TransaccionFactory` con un método estático `@staticmethod` llamado `crear_transaccion(canal: str, monto: float, **kwargs)` que retorne una instancia de la transacción adecuada según el canal (`"mercadopago"` o `"stripe"`). Lanza un `ValueError` si el canal no está soportado.
4. Escribe un flujo de prueba donde simules el pago de un carrito de compras de `$150.00` usando Stripe (en dólares) y un pago de `$4500.00` usando Mercado Pago, imprimiendo el resultado de `procesar_pago()` para cada transacción generada por la fábrica.

Implementa tu código a continuación.


In [ ]:
# Escribe la resolución aquí
# 1. Definir clase Transaccion
# ...

# 2. Definir clases hijas TransaccionMercadoPago y TransaccionStripe
# ...

# 3. Definir TransaccionFactory
# ...

# 4. Flujo de prueba
# ...
